In [3]:
import os
import cv2
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from skimage.feature import local_binary_pattern, hog
from tqdm import tqdm


# === Hàm check nhãn từ tên file
def label_check(filename):
    filename = filename.lower()
    if 'fake' in filename:
        return 0
    elif 'real' in filename:
        return 1
    else:
        return -1

# === Hàm xử lý từng ảnh riêng biệt (sẽ chạy song song)
def process_single_image(full_path, label):
    try:
        img_gray = cv2.imread(full_path, cv2.IMREAD_GRAYSCALE)
        if img_gray is None:
            print(f"Không đọc được ảnh: {full_path}")
            return None

        img_resized = cv2.resize(img_gray, (300, 300), interpolation=cv2.INTER_AREA)

        
        # === FFT
        fft_image = np.fft.fft2(img_resized)
        fft_shifted = np.fft.fftshift(fft_image)
        magnitude_spectrum = np.log1p(np.abs(fft_shifted)).flatten()


        features_with_label = np.append(magnitude_spectrum, label)
        return features_with_label
        

    except Exception as e:
        print(f"Lỗi ảnh {full_path}: {e}")
        return None

# === Class trích xuất đặc trưng
class ExtractFeature:
    def __init__(self, path_origin, path_real, path_fake, label_image_csv):
        self.path = path_origin
        self.path_real = path_real.strip('/')
        self.path_fake = path_fake.strip('/')
        self.label_image = pd.read_csv(label_image_csv, header=None)

    def extract_all(self, num_jobs=4):
        args_list = []
        for filename in self.label_image[0]:
            label = label_check(filename)
            if label == -1:
                continue
            sub_path = self.path_fake if label == 0 else self.path_real
            full_path = os.path.join(self.path, sub_path, filename)
            args_list.append((full_path, label))

        print(f"{num_jobs} CPU...")

        results = Parallel(n_jobs=num_jobs)(
            delayed(process_single_image)(path, label)
            for path, label in tqdm(args_list)
        )

        # Lọc bỏ kết quả None
        data_extract = [r for r in results if r is not None]

        if not data_extract:
            print("Không có ảnh hợp lệ.")
            return pd.DataFrame()

        n_features = len(data_extract[0]) - 1
        columns = [f'f{i}' for i in range(n_features)] + ['label']
        return pd.DataFrame(data_extract, columns=columns)


In [4]:
path = '/kaggle/input/data-img-fake-real'
path_real = '/real'
path_fake = '/fake'
labels = 'labels.csv'

ex = ExtractFeature(path, path_real, path_fake, os.path.join(path, labels))
df = ex.extract_all(num_jobs=4)
df.head()


4 CPU...


100%|██████████| 3417/3417 [01:24<00:00, 40.31it/s] 


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,...,f89991,f89992,f89993,f89994,f89995,f89996,f89997,f89998,f89999,label
0,5.455321,6.803012,6.916387,7.019597,7.634278,7.234165,6.689819,6.610899,5.848204,6.873254,...,6.479335,5.476715,6.369600,6.943029,6.780847,4.482894,6.130951,7.052834,6.222040,0.0
1,5.564520,7.426788,7.509238,6.901346,6.642542,6.653502,6.484016,7.774878,6.894057,4.955522,...,7.179169,6.195820,6.904962,7.480102,5.364117,7.334429,7.826482,7.857921,6.698990,0.0
2,6.811244,7.074574,7.338983,5.463220,6.503653,7.156263,5.934570,6.067958,6.150453,7.060361,...,7.419872,6.891273,6.533821,6.884580,6.539111,6.051238,4.633889,7.473244,6.241400,0.0
3,7.754053,5.491039,7.841576,6.719530,7.148400,6.776356,7.475415,6.592669,7.079554,6.880515,...,7.399733,6.923318,5.949911,7.636295,7.625813,7.609264,7.557969,7.593189,6.494007,1.0
4,7.696667,7.836997,6.493022,7.175246,7.635408,7.573038,6.198133,7.591490,7.263664,8.124040,...,8.307859,6.059317,7.848695,7.411481,5.242786,7.150817,6.961332,5.870700,7.282219,1.0


In [5]:
df.to_csv('features_handcraft.csv',index=False)

In [6]:
X = df.drop('label', axis=1)
y = df['label']

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [8]:
from sklearn.decomposition import PCA

pca = PCA(n_components=512)
X_pca = pca.fit_transform(X_scaled)

In [10]:
# Gộp lại PCA features với nhãn
features_pca = pd.DataFrame(X_pca)
features_pca['label'] = y.values  # thêm cột label vào cuối

# Lưu ra file CSV chứa feature và labels
features_pca.to_csv('/kaggle/working/features_handcraft_pca.csv', index=False)
